In [1]:
import sys
import numpy as np
import meshplot as mp
from pathlib import Path

# Add parent directory to path for DDG imports
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from core.mesh_io import load_mesh
from operators.gradient import gradient
from operators.divergence import divergence
from operators.curl import curl_vector, curl_scalar
from operators.laplacian import laplacian_0
from operators.exterior_derivative import d0, d1

In [2]:
import meshplot as mp

In [3]:
import numpy as np
import meshplot as mp
from pathlib import Path

# If needed:
from core.mesh_io import load_mesh
from core.halfedge import HalfEdgeMesh

mesh_files = [Path("data/bunny.obj")]
mesh_results = {}

for mesh_file in mesh_files:
    print(f"\n{'='*70}")
    print(f"🔄 Processing: {mesh_file.name}")
    print(f"{'='*70}")
    
    try:
        # -------------------------------------------------
        # 1️⃣ Load mesh
        # -------------------------------------------------
        V, F = load_mesh(str(mesh_file))
        print(f"✓ Loaded mesh from {mesh_file.name}")
        
        # Ensure zero-based indexing
        if F.min() == 1:
            F = F - 1
        
        # -------------------------------------------------
        # 2️⃣ Build HalfEdge structure
        # -------------------------------------------------
        mesh = HalfEdgeMesh(V, F)
        print("✓ Built HalfEdge structure")
        
        # -------------------------------------------------
        # 3️⃣ Basic topological statistics
        # -------------------------------------------------
        n_verts = V.shape[0]
        n_faces = F.shape[0]
        n_edges = len(mesh.halfedges) // 2
        n_boundary = len(mesh.boundary_edges())
        euler_char = n_verts - n_edges + n_faces
        
        print(f"\n📊 MESH STATISTICS:")
        print(f"  • Vertices: {n_verts}")
        print(f"  • Faces: {n_faces}")
        print(f"  • Edges: {n_edges}")
        print(f"  • Boundary edges: {n_boundary}")
        print(f"  • Euler characteristic: {euler_char}")
        
        # -------------------------------------------------
        # 4️⃣ Geometric statistics
        # -------------------------------------------------
        # Compute all edge lengths properly
        e0 = np.linalg.norm(V[F[:,0]] - V[F[:,1]], axis=1)
        e1 = np.linalg.norm(V[F[:,1]] - V[F[:,2]], axis=1)
        e2 = np.linalg.norm(V[F[:,2]] - V[F[:,0]], axis=1)
        edge_lengths = np.concatenate([e0, e1, e2])
        
        bbox = V.max(axis=0) - V.min(axis=0)
        total_area = mesh.vertex_area_voronoi().sum()
        
        print(f"\n🌐 GEOMETRIC STATISTICS:")
        print(f"  • Bounding box: {bbox}")
        print(f"  • Total surface area: {total_area:.6f}")
        print(f"  • Edge length (min/mean/max): "
              f"{edge_lengths.min():.6f} / "
              f"{edge_lengths.mean():.6f} / "
              f"{edge_lengths.max():.6f}")
        
        # -------------------------------------------------
        # 5️⃣ Visualization
        # -------------------------------------------------
        print("\n🖼 Visualizing mesh...")
        mp.plot(V, F, shading={"wireframe": False})
        
        # -------------------------------------------------
        # 6️⃣ Store results
        # -------------------------------------------------
        mesh_results[mesh_file.name] = {
            'V': V,
            'F': F,
            'mesh': mesh,
            'n_verts': n_verts,
            'n_faces': n_faces,
            'n_edges': n_edges,
            'total_area': total_area
        }
        
    except Exception as e:
        print(f"❌ Error processing {mesh_file.name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*70}")
print(f"✅ Successfully loaded {len(mesh_results)} mesh(es)")
print(f"{'='*70}")


🔄 Processing: bunny.obj
✓ Loaded mesh from bunny.obj
✓ Built HalfEdge structure

📊 MESH STATISTICS:
  • Vertices: 3485
  • Faces: 6966
  • Edges: 10449
  • Boundary edges: 0
  • Euler characteristic: 2

🌐 GEOMETRIC STATISTICS:
  • Bounding box: [0.1557956 0.1543756 0.1207922]
  • Total surface area: 0.058213
  • Edge length (min/mean/max): 0.000864 / 0.004661 / 0.021076

🖼 Visualizing mesh...


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…


✅ Successfully loaded 1 mesh(es)


In [4]:
import meshplot as mp
from geometry.curvature import mean_curvature, gaussian_curvature, principal_curvatures
for mesh_name, data in mesh_results.items():
    print(f"\n{'='*70}")
    print(f"📐 CURVATURE ANALYSIS: {mesh_name}")
    print(f"{'='*70}")
    
    try:
        mesh = data['mesh']
        V = data['V']
        F = data['F']
        
        # -------------------------
        # Gaussian curvature
        # -------------------------
        K = gaussian_curvature(mesh)
        print(f"\n✓ Gaussian Curvature K:")
        print(f"  • Min: {K.min():.6f}")
        print(f"  • Mean: {K.mean():.6f}")
        print(f"  • Max: {K.max():.6f}")
        print(f"  • Std: {K.std():.6f}")
        
        # -------------------------
        # Mean curvature
        # -------------------------
        H = mean_curvature(mesh)
        print(f"\n✓ Mean Curvature H:")
        print(f"  • Min: {H.min():.6f}")
        print(f"  • Mean: {H.mean():.6f}")
        print(f"  • Max: {H.max():.6f}")
        print(f"  • Std: {H.std():.6f}")
        
        # -------------------------
        # Principal curvatures
        # -------------------------
        k1, k2 = principal_curvatures(mesh)
        print(f"\n✓ Principal Curvatures k₁, k₂:")
        print(f"  k₁ • Min: {k1.min():.6f}, Mean: {k1.mean():.6f}, Max: {k1.max():.6f}")
        print(f"  k₂ • Min: {k2.min():.6f}, Mean: {k2.mean():.6f}, Max: {k2.max():.6f}")
        
        # Store results
        data['curvature'] = {'K': K, 'H': H, 'k1': k1, 'k2': k2}
        
        # -------------------------
        # Proper Surface Visualization
        # -------------------------
        print("\n🖼 Visualizing curvature on surface...")
        
        mp.plot(V, F, c=K, shading={"wireframe": False})
        mp.plot(V, F, c=H, shading={"wireframe": False})
        mp.plot(V, F, c=k1, shading={"wireframe": False})
        
    except Exception as e:
        print(f"❌ Curvature error: {e}")


📐 CURVATURE ANALYSIS: bunny.obj

✓ Gaussian Curvature K:
  • Min: -154332.859714
  • Mean: 1171.949329
  • Max: 434839.347579
  • Std: 18894.792523

✓ Mean Curvature H:
  • Min: 0.109453
  • Mean: 70.044566
  • Max: 1051.060038
  • Std: 72.460648

✓ Principal Curvatures k₁, k₂:
  k₁ • Min: 3.143022, Mean: 139.807084, Max: 1975.599180
  k₂ • Min: -271.993321, Mean: 0.282047, Max: 443.986927

🖼 Visualizing curvature on surface...


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…

In [5]:
from geometry.shape_operator import principal_directions
k1_dirs, k2_dirs = principal_directions(mesh)

# Visualize a few arrows
idx = np.random.choice(mesh.n_vertices, size=500, replace=False)

mp.plot(
    data['V'], data['F'],
    return_plot=True
).add_lines(
    data['V'][idx],
    data['V'][idx] + 0.01 * k1_dirs[idx]
)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…

1

In [6]:
print("Orthogonality error:",
      np.mean(np.sum(k1_dirs * k2_dirs, axis=1)))
from geometry.shape_operator import compute_vertex_normals
normals = compute_vertex_normals(mesh)

print("Tangency error k1:",
      np.mean(np.sum(k1_dirs * normals, axis=1)))

Orthogonality error: 1.661858955377004e-18
Tangency error k1: -1.2991983355324573e-18


In [7]:
import meshplot as mp
import numpy as np
from scipy.sparse.linalg import norm
from operators.gradient import gradient
from operators.divergence import divergence
from operators.curl import curl_scalar, curl_vector
from operators.laplacian import laplacian_0
from pde.hodge_decomposition import hodge_decomposition, is_divergence_free, is_curl_free

for mesh_name, data in mesh_results.items():
    print(f"\n{'='*70}")
    print(f"➡️  VECTOR OPERATORS: {mesh_name}")
    print(f"{'='*70}")
    
    try:
        mesh = data['mesh']
        V = data['V']
        F = data['F']
        
        n_verts = data['n_verts']
        n_edges = mesh.n_edges
        
        # -------------------------------------------------
        # Test scalar field (distance from center)
        # -------------------------------------------------
        center = V.mean(axis=0)
        scalar_field = np.linalg.norm(V - center, axis=1)
        
        print("\n✓ Scalar test field created (radial distance)")
        
        # -------------------------------------------------
        # Gradient
        # -------------------------------------------------
        grad_f = gradient(mesh, scalar_field)
        
        print("\n✓ Gradient (0-form → 1-form)")
        print(f"  shape: {grad_f.shape} (expected {n_edges})")
        print(f"  range: [{grad_f.min():.6f}, {grad_f.max():.6f}]")
        
        # -------------------------------------------------
        # Divergence
        # -------------------------------------------------
        div_grad = divergence(mesh, grad_f)
        
        print("\n✓ Divergence (1-form → 0-form)")
        print(f"  shape: {div_grad.shape} (expected {n_verts})")
        print(f"  range: [{div_grad.min():.6f}, {div_grad.max():.6f}]")
        
        # -------------------------------------------------
        # Laplace–Beltrami
        # -------------------------------------------------
        lap_u = curl_scalar(mesh, scalar_field)
        
        print("\n✓ Laplace–Beltrami Δu")
        print(f"  shape: {lap_u.shape}")
        print(f"  range: [{lap_u.min():.6f}, {lap_u.max():.6f}]")
        
        # -------------------------------------------------
        # Critical identity check
        # -------------------------------------------------
        identity_error = np.linalg.norm(div_grad - lap_u)
        print(f"\n🔎 div(grad u) - Δu error: {identity_error:.6e}")
        
        # -------------------------------------------------
        # d1 d0 = 0 test
        # -------------------------------------------------
        from operators.exterior_derivative import d0, d1
        
        D0 = d0(mesh)
        D1 = d1(mesh)
        zero_test = D1 @ D0
        
        zero_norm = np.linalg.norm(
            zero_test.toarray() if hasattr(zero_test, "toarray") else zero_test
        )
        print(f"🔎 ||d1 d0||: {zero_norm:.6e}")
        
        # -------------------------------------------------
        # Proper 3D Visualization
        # -------------------------------------------------
        print("\n🖼 Visualizing scalar field on surface")
        mp.plot(V, F, c=scalar_field, shading={"wireframe": False})
        
        print("\n🖼 Visualizing Laplacian Δu")
        mp.plot(V, F, c=lap_u, shading={"wireframe": False})
        
        # Store
        data['vectors'] = {
            'scalar_field': scalar_field,
            'gradient': grad_f,
            'divergence': div_grad,
            'laplacian': lap_u
        }
        
    except Exception as e:
        print(f"❌ Vector operator error: {e}")
        import traceback
        traceback.print_exc()


➡️  VECTOR OPERATORS: bunny.obj

✓ Scalar test field created (radial distance)

✓ Gradient (0-form → 1-form)
  shape: (10449,) (expected 10449)
  range: [-0.008734, 0.008538]

✓ Divergence (1-form → 0-form)
  shape: (3485,) (expected 3485)
  range: [-4112091.669447, 2919696.602259]

✓ Laplace–Beltrami Δu
  shape: (3485,)
  range: [-2919696.602259, 4112091.669447]

🔎 div(grad u) - Δu error: 3.320585e+07
🔎 ||d1 d0||: 0.000000e+00

🖼 Visualizing scalar field on surface


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…


🖼 Visualizing Laplacian Δu


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.016860…